In [1]:
import numpy as np
import math

class TicTacToe:
    def __init__(self):
        self.row_count = 3
        self.column_count = 3
        self.action_size = self.row_count * self.column_count

    def get_initial_state(self):
        return np.zeros((self.row_count, self.column_count))

    def get_next_state(self, state, action, player):
        row = action // self.column_count
        column = action % self.column_count
        state[row, column] = player
        return state

    def get_valid_moves(self, state):
        return (state.reshape(-1) == 0).astype(np.uint8)

    def check_win(self, state, action):
        if action == None:
            return False

        row = action // self.column_count
        column = action % self.column_count
        player = state[row, column]

        return (
            np.sum(state[row, :]) == player * self.column_count
            or np.sum(state[:, column]) == player * self.row_count
            or np.sum(np.diag(state)) == player * self.row_count
            or np.sum(np.diag(np.flip(state, axis=0))) == player * self.row_count
        )

    def get_value_and_terminated(self, state, action):
        if self.check_win(state, action):
            return 1, True
        if np.sum(self.get_valid_moves(state)) == 0:
            return 0, True
        return 0, False

    def get_opponent(self, player):
        return -player

    def get_opponent_value(self, value):
        return -value

    def change_perspective(self, state, player):
        return state * player

    def get_encoded_state(self, state):
        encoded_state = np.stack(
            (state == -1, state==0, state==1)
        ).astype(np.float32)
        return encoded_state

In [ ]:
tictactoe = TicTacToe()
state = tictactoe.get_initial_state()
state[0][0] = 1
tictactoe.get_valid_moves(state)

array([0, 1, 1, 1, 1, 1, 1, 1, 1], dtype=uint8)

In [ ]:
class Node:
    def __init__(self, game, args ,state, parent, action_taken=None):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken

        self.expandable_moves = game.get_valid_moves(state)
        self.children = []

        self.visit_count = 0
        self.value_sum = 0

    def get_ucb(self, child):
        q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + 1.41 * math.sqrt(math.log(self.visit_count) / child.visit_count)

    def is_fully_expanded(self):
        return np.sum(self.expandable_moves) == 0 and len(self.children) > 0

    def select(self):
        best_ucb_val = -np.inf
        best_ucb = None

        for child in self.children:
            ucb_val = self.get_ucb(child)
            if ucb_val > best_ucb_val:
                best_ucb_val = ucb_val
                best_ucb = child
        return best_ucb

    def expand(self):
        action = np.random.choice(np.where(self.expandable_moves == 1)[0])
        self.expandable_moves[action] = 0

        child_state = self.state.copy()
        child_state = self.game.get_next_state(child_state, action, 1)
        child_state = self.game.change_perspective(child_state, player=-1)

        child = Node(self.game, self.args, child_state, self, action)
        self.children.append(child)
        return child

    def simulate(self):
        value, is_terminal = self.game.get_value_and_terminated(self.state, self.action_taken)
        value = self.game.get_opponent_value(value)

        if is_terminal:
            return value

        rollout_state = self.state.copy()
        rollout_player = 1
        while True:
            valid_moves = self.game.get_valid_moves(rollout_state)
            action = np.random.choice(np.where(valid_moves == 1)[0])
            rollout_state = self.game.get_next_state(rollout_state, action, rollout_player)
            value, is_terminal = self.game.get_value_and_terminated(rollout_state, action)

            if is_terminal:
                if rollout_player == -1:
                    value = self.game.get_opponent_value(value)
                return value
            rollout_player = self.game.get_opponent(rollout_player)

    def backpropagate(self, value):
        self.visit_count += 1
        self.value_sum += value

        value = self.game.get_opponent_value(value)

        if self.parent is not None:
            self.parent.backpropagate(value)

In [ ]:
class MCTS:
    def __init__(self, game, args):
        self.game = game
        self.args = args

    def search(self, state):
        root = Node(self.game, self.args, state, None)
        for search in range(self.args['num_search']):
            node = root

            while node.is_fully_expanded():
                node = node.select()

            value, is_terminal = self.game.get_value_and_terminated(node.state, node.action_taken)
            value = self.game.get_opponent_value(value)

            if not is_terminal:
                node = node.expand()
                value = node.simulate()

            node.backpropagate(value)

        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs

In [ ]:
tictactoe = TicTacToe()
state = tictactoe.get_initial_state()

args = {
   'num_search': 1000
}

mcts = MCTS(tictactoe, args)
probs = mcts.search(state)
probs

array([0.134, 0.082, 0.104, 0.036, 0.211, 0.058, 0.143, 0.08 , 0.152])

In [ ]:
game = TicTacToe()
state = tictactoe.get_initial_state()
is_terminated = False

currentPlayer = 1
mcts = MCTS(tictactoe, args)

while not is_terminated:
    valid_moves = game.get_valid_moves(state)
    print(valid_moves)
    action = int(input("Your Action: "))

    state = game.get_next_state(state, action, currentPlayer)
    value, is_terminal = game.get_value_and_terminated(state, action)

    if is_terminal:
        break

    currentPlayer = game.get_opponent(currentPlayer)

    #bot
    probs = mcts.search(state)
    prob = np.argmax(probs)

    state = game.get_next_state(state, prob, currentPlayer)
    value, is_terminal = game.get_value_and_terminated(state, prob)

    print(state)

    if is_terminal:
        break
    currentPlayer = game.get_opponent(currentPlayer)

print(currentPlayer)


[1 1 1 1 1 1 1 1 1]


Your Action:  0


[[ 1.  0.  0.]
 [ 0.  0.  0.]
 [-1.  0.  0.]]
[0 1 1 1 1 1 0 1 1]


Your Action:  1


[[ 1.  1. -1.]
 [ 0.  0.  0.]
 [-1.  0.  0.]]
[0 0 0 1 1 1 0 1 1]


Your Action:  8


[[ 1.  1. -1.]
 [ 0. -1.  0.]
 [-1.  0.  1.]]
-1


In [ ]:
print("#############################")

#############################


In [3]:
import torch
from torch import nn
import torch.nn.functional as F

In [4]:
#model architecture

class Model(nn.Module):
    def __init__(self, input_shape,hidden_layers ,action_size, numResBlocks):
        super().__init__()
        self.start = nn.Sequential(
            nn.Conv2d(3, hidden_layers, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_layers),
            nn.ReLU()
        )

        self.resNetBlocks = nn.Sequential(*[ResBlock(hidden_layers) for _ in range(numResBlocks)])
        #moduleList nur container
        #mit sequential gehts

        self.policyHead = nn.Sequential(
            nn.Conv2d(hidden_layers, 32, kernel_size =3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32*3*3, action_size)
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(hidden_layers, 3, kernel_size =3, padding = 1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3*3*3, 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.start(x)
        x = self.resNetBlocks(x)
        p = self.policyHead(x)
        v = self.valueHead(x)
        return p, v


class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self,x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x

In [5]:
tictactoe = TicTacToe()
state = tictactoe.get_initial_state()
enc_state = tictactoe.get_encoded_state(state)
enc_state.shape

(3, 3, 3)

In [6]:
#Alphazeros Monte Carlo Tree Search

class Node:
    def __init__(self, game, args ,state, parent=None, action_taken=None):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.children = []

        self.action_taken = action_taken
        self.expandable_moves = game.get_valid_moves(state)

        self.visit_count = 0
        self.value_sum = 0
        self.P = 0

    def get_ucb(self, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + self.args['C'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.P

    def is_fully_expanded(self):
        return len(self.children) > 0

    def select(self):
        best_ucb_value = -np.inf
        best_child = None

        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb_value:
                best_ucb_value = ucb
                best_child = child
        return best_child

    def expand(self, policy_dists):
        #ertselle für ejde mögliche distribution ein neues child
        for action,dist in enumerate(policy_dists):
            if dist != 0:
                child_state = self.state.copy()
                child_state = self.game.get_next_state(child_state, action, 1)
                child_state = self.game.change_perspective(child_state, player=-1)

                child = Node(self.game, self.args, child_state, self, action)
                child.P = dist
                self.children.append(child)

    def backpropagate(self, value):
        self.visit_count += 1
        self.value_sum += value

        value = self.game.get_opponent_value(value)

        if self.parent is not None:
            self.parent.backpropagate(value)

In [7]:
class MCTS:
    def __init__(self, game, args):
        self.game = game
        self.args = args
        self.model = Model(input_shape=3, hidden_layers=100,action_size= 9, numResBlocks=5).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)

    def setEval(self):
        self.model.eval()

    def setTrain(self):
        self.model.train()

    def updateModel(self, loss):
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    @torch.no_grad()
    def search(self, state):
        root = Node(self.game, self.args, state)

        for search in range(self.args['num_search']):
            node = root

            while node.is_fully_expanded():
                node = node.select()

            value, is_terminal = self.game.get_value_and_terminated(node.state, node.action_taken)
            value = self.game.get_opponent_value(value)

            if not is_terminal:
                input_t = torch.tensor(self.game.get_encoded_state(node.state), dtype=torch.float32).unsqueeze(-1).to(device)
                input_t = input_t.permute(3, 0, 1, 2)
                policy, value = self.model(input_t)
                policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()
                policy /= np.sum(policy)
                policy *= self.game.get_valid_moves(node.state)

                value = value.item()
                node.expand(policy)

            node.backpropagate(value)
        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs

In [8]:
from tqdm.notebook import trange
import random

if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"
print(device)

cuda


In [9]:
class AlphaZero:
    def __init__(self, game , args):
        self.game = game
        self.args = args
        self.mcts = MCTS(game, args)

        self.memory = []

    def selfPlay(self):
        memory = []
        player = 1
        state = self.game.get_initial_state()
        while True:
            state = self.game.change_perspective(state, player)
            action_probs = self.mcts.search(state)
            memory.append((state, action_probs, player))

            action = np.random.choice(self.game.action_size, p=action_probs)
            state = self.game.get_next_state(state, action, player)
            value, is_terminal = game.get_value_and_terminated(state, action)

            if is_terminal:
                returnMemory = []
                for hist_neutral_state, hist_action_probs, hist_player in memory:
                    hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                    returnMemory.append((
                        self.game.get_encoded_state(hist_neutral_state),
                        hist_action_probs,
                        hist_outcome
                    ))
                return returnMemory

            player = self.game.get_opponent(player)


    def train(self, memory):
        random.shuffle(memory)
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])] # Change to memory[batchIdx:batchIdx+self.args['batch_size']] in case of an error
            state, policy_targets, value_targets = zip(*sample)

            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)

            state = torch.tensor(state, dtype=torch.float32).to(device)
            out_policy, out_value = self.mcts.model(state)

            policy_targets = torch.tensor(policy_targets, dtype=torch.float32).to(device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32).to(device)
            policy_loss = F.cross_entropy(out_policy, policy_targets)
            value_loss = F.mse_loss(out_value, value_targets)

            loss = policy_loss + value_loss

            self.mcts.updateModel(loss)

    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []

            self.mcts.setEval()
            for selfplayiter in trange(self.args['selfplayiter']):
                memory += self.selfPlay()

            self.mcts.setTrain()
            for epoch in trange(self.args['num_epochs']):
                self.train(memory)

In [10]:
game = TicTacToe()
state = tictactoe.get_initial_state()

args = {
    'num_search':1000,
    'C': 1.41,
    'num_iterations': 3,
    'selfplayiter': 100,
    'num_epochs': 4,
    'batch_size': 64
}

alphaZero = AlphaZero(game,args)
alphaZero.learn()

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
torch.save(alphaZero.mcts.model.state_dict(), "model.pth")

In [12]:
game = TicTacToe()
state = tictactoe.get_initial_state()
is_terminated = False

args = {
    'num_search':1000,
    'C': 1.41
}

currentPlayer = 1
mcts = MCTS(game, args)
mcts.model.load_state_dict(torch.load("model.pth"))

while not is_terminated:
    valid_moves = game.get_valid_moves(state)
    print(valid_moves)
    action = int(input("Your Action: "))

    state = game.get_next_state(state, action, currentPlayer)
    value, is_terminal = game.get_value_and_terminated(state, action)

    if is_terminal:
        break

    currentPlayer = game.get_opponent(currentPlayer)

    #bot
    probs = mcts.search(state)
    prob = np.argmax(probs)

    state = game.get_next_state(state, prob, currentPlayer)
    value, is_terminal = game.get_value_and_terminated(state, prob)

    print(state)

    if is_terminal:
        break
    currentPlayer = game.get_opponent(currentPlayer)

print(currentPlayer)

[1 1 1 1 1 1 1 1 1]
Your Action: 0
[[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  0.]]
[0 1 1 1 0 1 1 1 1]
Your Action: 1
[[ 1.  1. -1.]
 [ 0. -1.  0.]
 [ 0.  0.  0.]]
[0 0 0 1 0 1 1 1 1]
Your Action: 3
[[ 1.  1. -1.]
 [ 1. -1.  0.]
 [-1.  0.  0.]]
-1


In [ ]:
print("asd")